# Synthetic Regime-Switch Experiment

Manuscript Section 4.1 — proves the temporal-modeling claim by training the proposed hierarchical transformer router and the MLP-pool baseline on a controlled regime-switch dataset where the best expert depends on the *ordered pair* of regimes within the clip.

This notebook is a **reproducibility package**, not an exploratory
scratchpad. Cells contain only shell calls (`!uv run python -m ...` /
`!make ...`) and one final rendering cell that emits the
manuscript-grade table/figure for this experiment from `src.reporting`.
All artifacts are also pushed to W&B.

Experiment config: [`configs/experiments/synthetic_v2.yaml`](../../configs/experiments/synthetic_v2.yaml). The full operating
protocol is in [`PROJECT_STATE.md`](../../PROJECT_STATE.md).


## 1. Setup — clone, `uv` env, secrets


In [1]:
%%bash
set -euo pipefail
if [ ! -d s2t-tr-dev ]; then
  git clone -b main_v2 https://github.com/huseyin-karaca/s2t-tr-dev
fi
cd s2t-tr-dev
if ! command -v uv >/dev/null 2>&1; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
  export PATH="$HOME/.cargo/bin:$PATH"
fi
uv venv --python 3.10 --no-managed-python
uv sync


Cloning into 's2t-tr-dev'...
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 214 packages in 11ms
Prepared 210 packages in 16.32s
Installed 210 packages in 162ms
 + absl-py==2.4.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.13.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + async-timeout==5.0.1
 + attrs==26.1.0
 + audioread==3.1.0
 + babel==2.18.0
 + beautifulsoup4==4.14.3
 + bleach==6.3.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.3.2
 + comm==0.2.3
 + contourpy==1.3.2
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.3
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + datasets==3.6.0
 + debugpy==1.8.20
 + decorator==5.2.1
 + defusedxml==0.7.1
 + dill==0.3.8
 + dotenv==0.9.9
 +

In [2]:
%cd s2t-tr-dev



/content/s2t-tr-dev


In [3]:
from google.colab import files
files.view("/content/s2t-tr-dev")

<IPython.core.display.Javascript object>

In [10]:
!uv run python -m src.scripts.colab_helpers



{'WANDB_API_KEY': False, 'WANDB_ENTITY': False, 'HF_TOKEN': False}


In [11]:

from google.colab import userdata
import os
os.environ['GITHUB_TOKEN'] = userdata.get('GitHubPAT')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# W&B kimlik doğrulaması için eklendi
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

!gh auth status

github.com
  ✓ Logged in to github.com account huseyin-karaca (GITHUB_TOKEN)
  - Active account: true
  - Git operations protocol: https
  - Token: github_pat_11AUNXCOA01Derj8bnLEVE_***********************************************************


## 2. Data


In [8]:
!mkdir -p data/processed/synthetic_regime_switch
!uv run python -m src.data.synthetic \
    --output-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --num-samples 10000 --frame-length 128 --num-regimes 4 \
    --regime-dim 32 --noise-std 0.5 --feature-dtype float16 \
    --write-batch-size 500 --seed 42


2026-05-01 18:01:15,050 [INFO] __main__: Generating 10000 clips → data/processed/synthetic_regime_switch/combined_features.parquet (T=128, R=4, regime_dim=32, dims={'hubert': 1024, 'whisper': 512, 'wav2vec2': 1024}, dtype=float16)
2026-05-01 18:01:15,050 [INFO] __main__: Embedding payload: 6.55 GB total, peak per write batch: 0.33 GB (write_batch_size=500)
2026-05-01 18:01:33,584 [INFO] __main__:   wrote 2000 / 10000 clips
2026-05-01 18:01:51,239 [INFO] __main__:   wrote 4000 / 10000 clips
2026-05-01 18:02:08,892 [INFO] __main__:   wrote 6000 / 10000 clips
2026-05-01 18:02:26,623 [INFO] __main__:   wrote 8000 / 10000 clips
2026-05-01 18:02:46,842 [INFO] __main__:   wrote 10000 / 10000 clips
2026-05-01 18:02:46,843 [INFO] __main__: Sanity stats — random WER: 0.3785, oracle WER: 0.1531, per-expert mean WER: {'hubert': 0.43208104372024536, 'whisper': 0.344176709651947, 'wav2vec2': 0.3591602146625519}
2026-05-01 18:02:46,843 [INFO] __main__: Done.


## 3. Run the experiment

All knobs live in `configs/experiments/synthetic_v2.yaml`. Per the
SSOT rule, do not pass overrides on the CLI for manuscript runs;
if a parameter must change, create `<name>_v3.yaml` (or higher).


In [16]:
! git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 837 bytes | 837.00 KiB/s, done.
From https://github.com/huseyin-karaca/s2t-tr-dev
   5424c3d..ec2a541  main_v2    -> origin/main_v2
Updating 5424c3d..ec2a541
Fast-forward
 src/utils/checkpoint.py | 11 ++++++++++-
 1 file changed, 10 insertions(+), 1 deletion(-)


In [17]:
!uv run python -m src.experiments.run experiments=synthetic_v2


2026-05-01 18:18:52 INFO     __main__:main:128 | === experiment_metadata ===
2026-05-01 18:18:52 INFO     __main__:main:130 |   name: synthetic_v2
2026-05-01 18:18:52 INFO     __main__:main:130 |   parent: synthetic
2026-05-01 18:18:52 INFO     __main__:main:130 |   status: active
2026-05-01 18:18:52 INFO     __main__:main:130 |   created: 2026-05-01
2026-05-01 18:18:52 INFO     __main__:main:130 |   author: huseyin-karaca
2026-05-01 18:18:52 INFO     __main__:main:130 |   deprecated_for: None
2026-05-01 18:18:52 INFO     __main__:main:130 |   interpretation: The v1 synthetic config was a 2-epoch smoke run inheriting AMI-flavored
defaults. It established that the pipeline (data generation, training,
eval) is wired correctly, but the resulting numbers (final WER ~0.5
on the proposed router, far from converged) are not manuscript-grade.

2026-05-01 18:18:52 INFO     __main__:main:130 |   rationale: For the manuscript Tab. I (synthetic regime-switch test results) we
need converged numbers

### 3a. (Optional) Sweep over $R$

Generates the manuscript Fig. `synthetic_sweep` by re-running
the pipeline at $R \in \{2,3,4,6,8\}$. Comment in if you want
the full sweep; defaults are tuned for a Colab T4.


In [18]:
!uv run python -m src.experiments.sweep run \
     --output-dir reports/sweeps/synthetic_R \
     --r-values 2,3,4,6,8 \
     --num-samples 5000 --frame-length 128 \
     --max-epochs 30 --batch-size 32 --seed 42
!uv run python -m src.experiments.sweep figure \
     --results reports/sweeps/synthetic_R/sweep_results.json \
     --output-path reports/manuscript/figures/auto/synthetic_v2/synthetic_sweep.pdf


2026-05-01 18:25:10 INFO     __main__:run_sweep:186 | Sweep R values: [2, 3, 4, 6, 8] — output dir: reports/sweeps/synthetic_R
2026-05-01 18:25:10 INFO     src.experiments._common:run:45 | >>> generate synthetic parquet (R=2) — /content/s2t-tr-dev/.venv/bin/python3 -m src.data.synthetic --output-path reports/sweeps/synthetic_R/R2/combined_features.parquet --num-samples 5000 --frame-length 128 --num-regimes 2 --regime-dim 32 --noise-std 0.5 --feature-dtype float16 --write-batch-size 500 --seed 42
2026-05-01 18:25:10,554 [INFO] __main__: Generating 5000 clips → reports/sweeps/synthetic_R/R2/combined_features.parquet (T=128, R=2, regime_dim=32, dims={'hubert': 1024, 'whisper': 512, 'wav2vec2': 1024}, dtype=float16)
2026-05-01 18:25:10,554 [INFO] __main__: Embedding payload: 3.28 GB total, peak per write batch: 0.33 GB (write_batch_size=500)
2026-05-01 18:25:28,894 [INFO] __main__:   wrote 2000 / 5000 clips
2026-05-01 18:25:46,791 [INFO] __main__:   wrote 4000 / 5000 clips
2026-05-01 18:25

In [19]:
!git pull
!uv run python -m src.experiments.sweep run \
     --output-dir reports/sweeps/synthetic_R \
     --r-values 2,3,4,6,8 \
     --num-samples 5000 --frame-length 128 \
     --max-epochs 30 --seed 42

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.19 KiB | 1.19 MiB/s, done.
From https://github.com/huseyin-karaca/s2t-tr-dev
   ec2a541..c9bb532  main_v2    -> origin/main_v2
Updating ec2a541..c9bb532
Fast-forward
 src/experiments/sweep.py | 37 ++++++++++++++++++++++++++++++++++---
 1 file changed, 34 insertions(+), 3 deletions(-)
2026-05-01 18:36:18 INFO     __main__:run_sweep:212 | Sweep R values: [2, 3, 4, 6, 8] — output dir: reports/sweeps/synthetic_R
2026-05-01 18:36:18 INFO     src.experiments._common:run:45 | >>> generate synthetic parquet (R=2) — /content/s2t-tr-dev/.venv/bin/python3 -m src.data.synthetic --output-path reports/sweeps/synthetic_R/R2/combined_features.parquet --num-samples 5000 --frame-length 128 --num-regimes 2 --regime-dim 32 --noise-std 0.5 --feature-dtype float16 --write-batch-s

## 4. Render manuscript deliverables

Builds Markdown + LaTeX tables and the companion figures from
the `main_results.json` aggregator written by step 3, then pushes
everything to Weights & Biases as a `results-table` /
`results-figures` artifact tied to the current git commit.


In [21]:
!uv run python -m src.reporting.tables \
    --results reports/main_results/synthetic_v2/main_results.json \
    --output-dir reports/manuscript/figures/auto/synthetic_v2 \
    --push-wandb


2026-05-01 18:46:32 INFO     __main__:render_main_results_table:87 | Rendered table to reports/manuscript/figures/auto/synthetic_v2/results.md and reports/manuscript/figures/auto/synthetic_v2/results.tex
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hkaraca to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.

m
m
m
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/s2t-tr-dev/wandb/run-20260501_184632-qdt7qoht
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reporting-tables-synthetic_v2
wandb: ⭐️ View project at https://wandb.ai/hkaraca/s2t-tr-dev
wandb: 🚀 View run at https://wandb.ai/hkaraca/s2t-tr-dev/runs/qdt7qoht


m

m

m

m


m


m


m


m


m



m



m



m



m



m



m



m



m



m



m
m
m
m
m
m
m
m
m
m
m




m


In [22]:
!uv run python -m src.reporting.figures \
    --results reports/main_results/synthetic_v2/main_results.json \
    --output-dir reports/manuscript/figures/auto/synthetic_v2 \
    --push-wandb


2026-05-01 18:47:00 INFO     __main__:render_cli:176 | Rendered figure: reports/manuscript/figures/auto/synthetic_v2/method_wer_bars.pdf
2026-05-01 18:47:00 INFO     __main__:render_cli:176 | Rendered figure: reports/manuscript/figures/auto/synthetic_v2/selection_frequency.pdf
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hkaraca to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.

m
m
m
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/s2t-tr-dev/wandb/run-20260501_184701-bpojmhsb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reporting-figures-synthetic_v2
wandb: ⭐️ View project at https://wandb.ai/hkaraca/s2t-tr-dev
wandb: 🚀 View run at https://wandb.ai/hkaraca/s2t-tr-dev/runs/bpojmhsb



m


m


m


m

m

m

m

m

m




## 5. Display the rendered deliverables


In [23]:
from src.scripts.colab_helpers import display_deliverables
display_deliverables('synthetic_v2')


ModuleNotFoundError: No module named 'loguru'

In [ ]:
# Optional: free the Colab GPU once finished.
# from google.colab import runtime
# runtime.unassign()
